In [1]:
import numpy as np
import pandas as pd
from surprise import Dataset, Reader, KNNBaseline, SVD, accuracy
from surprise.model_selection import GridSearchCV, train_test_split

In [2]:
MIN_REVIEWS = 20
TOP_K = 10
RNG = 42

# Filtering users by number of reviews done and items by number of reviews received

In [3]:
reviews = pd.read_csv("reviews_clean.csv")
meta = pd.read_csv("meta_clean.csv")

user_counts = reviews['user_id'].value_counts()
item_counts = reviews['parent_asin'].value_counts()

suitable_users = user_counts[user_counts >= MIN_REVIEWS].index
suitable_items = item_counts[item_counts >= MIN_REVIEWS].index

ratings = reviews[
    reviews['user_id'].isin(suitable_users) & reviews['parent_asin'].isin(suitable_items)
].copy()
ratings['rating'] = ratings['rating'].astype(float)

display(f'Min_Reviews: {MIN_REVIEWS}')
print(f'Righe originali: {len(reviews)}')
print(f'Righe dopo filtro: {len(ratings)}')
print(f'Utenti considerati: {ratings["user_id"].nunique()}')
print(f'Item considerati: {ratings["parent_asin"].nunique()}')
display(ratings.head())

'Min_Reviews: 20'

Righe originali: 4624175
Righe dopo filtro: 164819
Utenti considerati: 5718
Item considerati: 19310


,rating,parent_asin,user_id
92,5.0,B00LINQ1HY,AFW2PDT3AMT4X3PYQG7FJZH5FXFA
93,3.0,B00O2GW3EO,AFW2PDT3AMT4X3PYQG7FJZH5FXFA
94,4.0,B00C9UND8U,AFW2PDT3AMT4X3PYQG7FJZH5FXFA
95,3.0,B007XJ448A,AFW2PDT3AMT4X3PYQG7FJZH5FXFA
96,5.0,B0041CASX2,AFW2PDT3AMT4X3PYQG7FJZH5FXFA


# Data overview

In [4]:
user_item_matrix = ratings.pivot_table(
    index='user_id',
    columns='parent_asin',
    values='rating',
    aggfunc='mean'
)

reader = Reader(rating_scale=(1, 5))
dataset_surprise = Dataset.load_from_df(ratings[['user_id', 'parent_asin', 'rating']], reader)

n_users, n_items = user_item_matrix.shape
sparsity = 1 - (ratings.shape[0] / (n_users * n_items))
print(f'Dimensione matrice user x item: {n_users} x {n_items}')
print(f'Sparsita: {sparsity:.4f}')
user_item_matrix.head()

Dimensione matrice user x item: 5718 x 19310
Sparsita: 0.9985


parent_asin,0375869026,0399588175,0439381673,0692681663,0700026398,0700026657,0700099867,0700099956,0744014298,0744018706,...,B0CFDHCM9Q,B0CFXKTL31,B0CGMDJJX7,B0CGS7YCZ4,B0CGY5ZT55,B0CH12MZQL,B0CHJC16TX,B0CHP8VT5N,B0CHRQXBWC,B0CHW19XX3
user_id,,,,,,,,,,,,,,,,,,,,,
AE2324DZTYWS3LKNTEKJJZYHSQXQ,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AE23OKGK5L7UNO32OVCQKOEXKIBQ,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AE23PXMECDNZRSKAQV6CAIEN67UQ,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AE25Y2LOSEKTPUJFDPWYJNYCQ7EQ,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AE25ZDXYBK3LHKCZ7XUODANPME4A,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Grid searching optimal KNN model

In [5]:
knn_param_grid = {
    'k': [15, 25, 35],
    'sim_options': {
        'name': ['msd', 'pearson', 'cosine'],
        'user_based': [False, True],
    },
    'verbose': [False],
}

knn_gs = GridSearchCV(
    KNNBaseline,
    knn_param_grid,
    measures=['mse', 'rmse'],
    cv=5,
    n_jobs=1,
)

knn_gs.fit(dataset_surprise)

In [6]:
knn_results = pd.DataFrame(knn_gs.cv_results)
knn_results['k'] = knn_results['param_k'].astype(int)
knn_results['similarity'] = knn_results['param_sim_options'].apply(lambda d: d['name'])
knn_results['user_based'] = knn_results['param_sim_options'].apply(lambda d: d['user_based'])

knn_metrics = knn_results[[
    'k',
    'similarity',
    'user_based',
    'mean_test_mse',
    'mean_test_rmse',
    'rank_test_rmse'
]].sort_values(['rank_test_rmse', 'mean_test_mse']).reset_index(drop=True)

best_knn_params = knn_gs.best_params['rmse']

display(knn_metrics)
print(f"Best KNN params: {best_knn_params}")
print(f"Best KNN MSE: {knn_gs.best_score['mse']:.6f}")
print(f"Best KNN RMSE: {knn_gs.best_score['rmse']:.6f}")

,k,similarity,user_based,mean_test_mse,mean_test_rmse,rank_test_rmse
0,25,msd,False,1.021999,1.010918,1
1,35,msd,False,1.022313,1.011073,2
2,15,msd,False,1.023137,1.011481,3
3,35,pearson,False,1.026715,1.013260,4
4,25,pearson,False,1.026759,1.013282,5
5,15,pearson,False,1.026971,1.013387,6
6,35,cosine,False,1.032673,1.016181,7
7,25,cosine,False,1.033334,1.016506,8
8,15,cosine,False,1.036400,1.018013,9
9,35,pearson,True,1.047753,1.023589,10


Best KNN params: {'k': 25, 'sim_options': {'name': 'msd', 'user_based': False}, 'verbose': False}
Best KNN MSE: 1.021999
Best KNN RMSE: 1.010918


# Evaluating optimal KNN model

In [9]:
trainset_eval, testset_eval = train_test_split(
    dataset_surprise,
    test_size=0.2,
    random_state=RNG
)

best_knn_eval_algo = KNNBaseline(**best_knn_params)
best_knn_eval_algo.fit(trainset_eval)
knn_eval_predictions = best_knn_eval_algo.test(testset_eval)

knn_eval_mse = accuracy.mse(knn_eval_predictions, verbose=False)
knn_eval_rmse = accuracy.rmse(knn_eval_predictions, verbose=False)

display(pd.DataFrame([{
    'modello': 'KNN rank 1',
    'MSE': knn_eval_mse,
    'RMSE': knn_eval_rmse
}]))

,modello,MSE,RMSE
0,KNN rank 1,1.037704,1.018678


# Helper functions: missing rating filler and top-K recommendations

In [10]:
def fill_missing_with_model(algo, trainset, base_matrix):
    filled_values = base_matrix.to_numpy(dtype=np.float32).copy()
    uid_to_idx = {uid: idx for idx, uid in enumerate(base_matrix.index)}
    iid_to_idx = {iid: idx for idx, iid in enumerate(base_matrix.columns)}

    anti_testset = trainset.build_anti_testset()
    predictions = algo.test(anti_testset)

    for pred in predictions:
        filled_values[uid_to_idx[pred.uid], iid_to_idx[pred.iid]] = pred.est

    return pd.DataFrame(filled_values, index=base_matrix.index, columns=base_matrix.columns)


def build_top_k_recommendations(filled_matrix, original_matrix, item_titles, top_k=10):
    users = filled_matrix.index.to_numpy()
    items = filled_matrix.columns.to_numpy()
    filled_values = filled_matrix.to_numpy(dtype=np.float32)
    seen_mask = ~np.isnan(original_matrix.to_numpy(dtype=np.float32))

    rows = []
    for u_idx, uid in enumerate(users):
        scores = filled_values[u_idx].copy()
        scores[seen_mask[u_idx]] = -np.inf
        ranked_idx = np.argsort(scores)[::-1][:top_k]

        for rank, item_idx in enumerate(ranked_idx, start=1):
            iid = items[item_idx]
            rows.append({
                'user_id': uid,
                'rank': rank,
                'parent_asin': iid,
                'title': item_titles.loc[iid],
                'pred_rating': float(scores[item_idx])
            })

    return pd.DataFrame(rows)

In [11]:
full_trainset = dataset_surprise.build_full_trainset()

# Predicting missing values with the best KNN model and saving the filled matrix

In [13]:
best_knn_full_algo = KNNBaseline(**best_knn_params)
best_knn_full_algo.fit(full_trainset)

knn_filled_matrix = fill_missing_with_model(best_knn_full_algo, full_trainset, user_item_matrix)
knn_filled_matrix.to_csv('cf_filled_matrix_knn.csv', index=True)
display(knn_filled_matrix.head())
print("\n statistiche dei ratings (comprende sia quelli gia esistenti che quelli predetti)")
display(knn_filled_matrix.stack().describe())

parent_asin,0375869026,0399588175,0439381673,0692681663,0700026398,0700026657,0700099867,0700099956,0744014298,0744018706,...,B0CFDHCM9Q,B0CFXKTL31,B0CGMDJJX7,B0CGS7YCZ4,B0CGY5ZT55,B0CH12MZQL,B0CHJC16TX,B0CHP8VT5N,B0CHRQXBWC,B0CHW19XX3
user_id,,,,,,,,,,,,,,,,,,,,,
AE2324DZTYWS3LKNTEKJJZYHSQXQ,3.734414,4.976954,3.684365,3.581902,3.603804,3.072520,3.476730,3.650485,3.569674,3.860896,...,3.684745,3.848683,3.791247,3.491486,3.654636,3.590630,3.916611,2.628414,3.737450,3.706458
AE23OKGK5L7UNO32OVCQKOEXKIBQ,4.768545,4.851526,4.490479,4.388016,4.759461,4.547895,4.556417,4.848371,4.321770,4.667010,...,4.821365,4.028192,4.409298,4.297600,3.954100,4.396744,4.932520,4.384322,4.543564,4.512572
AE23PXMECDNZRSKAQV6CAIEN67UQ,3.925738,3.963164,3.875690,3.324527,3.871099,3.192530,3.719892,1.752127,3.760998,4.052220,...,3.876069,4.040007,3.794508,3.682810,3.224602,3.781955,4.159773,2.938008,3.928774,3.897782
AE25Y2LOSEKTPUJFDPWYJNYCQ7EQ,4.842627,4.702863,4.615388,4.688639,4.786512,4.517406,4.407753,4.682682,4.500697,4.898983,...,4.615768,4.779706,4.709921,4.392464,4.871111,4.781666,5.000000,4.663189,4.668473,4.637481
AE25ZDXYBK3LHKCZ7XUODANPME4A,1.006430,2.012131,1.924657,1.000000,1.009644,1.427499,1.000000,1.307231,1.440234,1.353755,...,1.000000,1.192569,1.843476,1.000000,1.000000,1.000000,1.357803,1.102651,1.977741,1.946749



 statistiche dei ratings (comprende sia quelli gia esistenti che quelli predetti)


count    1.104146e+08
mean     4.250508e+00
std      6.302798e-01
min      1.000000e+00
25%      4.010659e+00
50%      4.370240e+00
75%      4.662065e+00
max      5.000000e+00
dtype: float64

In [14]:
item_titles = (
    meta.set_index('parent_asin')['title']
    .reindex(user_item_matrix.columns)
)

# Top-k recommendations for each users (KNN)

In [15]:
knn_top10 = build_top_k_recommendations(
    knn_filled_matrix,
    user_item_matrix,
    item_titles,
    top_k=TOP_K
)

print(f'Utenti: {knn_top10["user_id"].nunique()}')
print(f'Record raccomandazioni KNN: {len(knn_top10)}')
display(knn_top10.head(30))

Utenti: 5718
Record raccomandazioni KNN: 57180


,user_id,rank,parent_asin,title,pred_rating
0,AE2324DZTYWS3LKNTEKJJZYHSQXQ,1,B0019952BI,Heroes of Might and Magic III & IV Complete Co...,5.0
1,AE2324DZTYWS3LKNTEKJJZYHSQXQ,2,B0031ACJMS,Escape Rosecliff Island - PC,5.0
2,AE2324DZTYWS3LKNTEKJJZYHSQXQ,3,B0009OO6Y8,Spartan: Total Warrior - PlayStation 2,5.0
3,AE2324DZTYWS3LKNTEKJJZYHSQXQ,4,B001EYUU7O,NFL 2K3 - Xbox,5.0
4,AE2324DZTYWS3LKNTEKJJZYHSQXQ,5,B00P1OYN0S,Tomee 256KB Memory Card for N64,5.0
5,AE2324DZTYWS3LKNTEKJJZYHSQXQ,6,B0000CE1J2,The Command and Conquer Collection - PC,5.0
6,AE2324DZTYWS3LKNTEKJJZYHSQXQ,7,B000I93SGQ,Whisper Fan 360 Green,5.0
7,AE2324DZTYWS3LKNTEKJJZYHSQXQ,8,B00004ZBOE,Commandos 2: Men of Courage - PC,5.0
8,AE2324DZTYWS3LKNTEKJJZYHSQXQ,9,B00006IR4T,Indiana Jones and the Emperor's Tomb - PC,5.0
9,AE2324DZTYWS3LKNTEKJJZYHSQXQ,10,B00004YZAO,Namco Museum Vol. 1 (PlayStation),5.0


# Grid Searching Optimal SVD (Matrix Factorization) Configuration

In [16]:
svd_param_grid = {
    'n_factors': [70, 100, 130, 160],
    'n_epochs': [20, 30, 40],
    'random_state': [RNG],
}

svd_gs = GridSearchCV(
    SVD,
    svd_param_grid,
    measures=['mse', 'rmse'],
    cv=5,
    n_jobs=-1
)
svd_gs.fit(dataset_surprise)

In [17]:
svd_results = pd.DataFrame(svd_gs.cv_results)
svd_metrics = svd_results[[
    'param_n_factors',
    'param_n_epochs',
    'mean_test_mse',
    'mean_test_rmse',
    'rank_test_rmse'
]].rename(columns={
    'param_n_factors': 'n_factors',
    'param_n_epochs': 'n_epochs',
}).sort_values(['rank_test_rmse', 'mean_test_mse']).reset_index(drop=True)

best_svd_params = svd_gs.best_params['rmse']

display(svd_metrics)
print(f"Best SVD params: {best_svd_params}")
print(f"Best SVD MSE: {svd_gs.best_score['mse']:.6f}")
print(f"Best SVD RMSE: {svd_gs.best_score['rmse']:.6f}")

,n_factors,n_epochs,mean_test_mse,mean_test_rmse,rank_test_rmse
0,100,30,0.929667,0.964176,1
1,70,30,0.930509,0.964608,2
2,100,20,0.930572,0.964645,3
3,70,20,0.930756,0.964738,4
4,160,30,0.932348,0.965563,5
5,130,30,0.933130,0.965968,6
6,130,20,0.933300,0.966057,7
7,160,20,0.933305,0.966060,8
8,100,40,0.936531,0.967728,9
9,160,40,0.936972,0.967954,10


Best SVD params: {'n_factors': 100, 'n_epochs': 30, 'random_state': 42}
Best SVD MSE: 0.929667
Best SVD RMSE: 0.964176


# Evaluating Best SVD model

In [18]:
best_svd_eval_algo = SVD(**best_svd_params)
best_svd_eval_algo.fit(trainset_eval)
svd_eval_predictions = best_svd_eval_algo.test(testset_eval)

svd_eval_mse = accuracy.mse(svd_eval_predictions, verbose=False)
svd_eval_rmse = accuracy.rmse(svd_eval_predictions, verbose=False)

display(pd.DataFrame([{
    'modello': 'SVD ottimale',
    'MSE': svd_eval_mse,
    'RMSE': svd_eval_rmse
}]))

,modello,MSE,RMSE
0,SVD ottimale,0.932949,0.965893


# Predicting missing values with the best SVD model and saving the filled matrix

In [ ]:
best_svd_full_algo = SVD(**best_svd_params)
best_svd_full_algo.fit(full_trainset)

svd_filled_matrix = fill_missing_with_model(best_svd_full_algo, full_trainset, user_item_matrix)
svd_filled_matrix.to_csv('cf_filled_matrix_svd.csv', index=True)
display(svd_filled_matrix.head())
print("\n statistiche dei ratings (comprende sia quelli gia esistenti che quelli predetti)")
display(svd_filled_matrix.stack().describe())

parent_asin,0375869026,0399588175,0439381673,0692681663,0700026398,0700026657,0700099867,0700099956,0744014298,0744018706,...,B0CFDHCM9Q,B0CFXKTL31,B0CGMDJJX7,B0CGS7YCZ4,B0CGY5ZT55,B0CH12MZQL,B0CHJC16TX,B0CHP8VT5N,B0CHRQXBWC,B0CHW19XX3
user_id,,,,,,,,,,,,,,,,,,,,,
AE2324DZTYWS3LKNTEKJJZYHSQXQ,3.597183,3.471705,3.009663,3.251816,3.322934,3.040936,3.368697,3.027313,3.052795,3.416764,...,3.679438,3.646031,3.253090,3.277505,3.927919,3.367787,3.624218,3.263091,3.375110,3.390491
AE23OKGK5L7UNO32OVCQKOEXKIBQ,4.683881,4.627562,4.366667,4.476957,4.481173,4.096402,4.301665,4.336919,4.361231,4.680383,...,4.498158,4.971791,4.524440,4.259621,4.680042,4.432800,4.818347,4.497356,4.476310,4.445241
AE23PXMECDNZRSKAQV6CAIEN67UQ,3.850560,3.750058,3.575102,3.474054,3.491356,3.607107,3.241668,3.718750,3.440243,4.274238,...,3.609052,3.923343,3.700168,3.413892,3.751727,3.499229,4.042907,3.749288,3.993311,3.694973
AE25Y2LOSEKTPUJFDPWYJNYCQ7EQ,4.870823,4.971504,4.686490,4.669246,4.850224,4.267315,4.562845,4.786350,4.756733,4.935230,...,4.853590,5.000000,4.632627,4.561975,4.847180,4.703556,5.000000,4.793759,4.872095,4.820868
AE25ZDXYBK3LHKCZ7XUODANPME4A,1.661305,1.646191,1.643846,1.410143,1.584640,1.403772,1.523527,1.477309,1.179690,2.026925,...,1.638190,1.904178,1.759217,1.282204,1.732915,1.449986,1.985530,1.539873,2.015205,1.891818



 statistiche dei ratings (comprende sia quelli gia esistenti che quelli predetti)


count    1.104146e+08
mean     4.248318e+00
std      5.557700e-01
min      1.000000e+00
25%      3.927985e+00
50%      4.331689e+00
75%      4.672359e+00
max      5.000000e+00
dtype: float64

# Top-k recommendations for each users (SVD)

In [20]:
svd_top10 = build_top_k_recommendations(
    svd_filled_matrix,
    user_item_matrix,
    item_titles,
    top_k=TOP_K
)


print(f'Utenti: {svd_top10["user_id"].nunique()}')
print(f'Record raccomandazioni SVD: {len(svd_top10)}')
display(svd_top10.head(30))

Utenti: 5718
Record raccomandazioni SVD: 57180


,user_id,rank,parent_asin,title,pred_rating
0,AE2324DZTYWS3LKNTEKJJZYHSQXQ,1,B09MFBJJQJ,Devil May Cry,4.374909
1,AE2324DZTYWS3LKNTEKJJZYHSQXQ,2,B0002BJQOI,18 Wheels of Steel: Pedal to the Metal - PC,4.323215
2,AE2324DZTYWS3LKNTEKJJZYHSQXQ,3,B000035XR9,Sega Genesis Core System 2 - Video Game Console,4.320335
3,AE2324DZTYWS3LKNTEKJJZYHSQXQ,4,B0002A6CQ4,Resident Evil 4 - Gamecube,4.318952
4,AE2324DZTYWS3LKNTEKJJZYHSQXQ,5,B00HLWIS72,Skylanders SWAP Force Sheep Wreck Island Adven...,4.272205
5,AE2324DZTYWS3LKNTEKJJZYHSQXQ,6,B001ELJPPA,Xbox Controller S- Green,4.234665
6,AE2324DZTYWS3LKNTEKJJZYHSQXQ,7,B07B62LMC1,Victor Vran: Overkill Edition - Nintendo Switch,4.216267
7,AE2324DZTYWS3LKNTEKJJZYHSQXQ,8,B0006BK58U,Forza Motorsport,4.215761
8,AE2324DZTYWS3LKNTEKJJZYHSQXQ,9,B001QCWRZC,Dragon Age: Origins Digital Deluxe Edition [Do...,4.210478
9,AE2324DZTYWS3LKNTEKJJZYHSQXQ,10,B00ZSDTIGG,DualShock 4 Wireless Controller for PlayStatio...,4.201390


# KNN vs SVD: (MSE, RMSE, recommendations)

In [21]:
comparison_metrics = pd.DataFrame([
    {
        'model': 'KNN (best)',
        'cv_mse': knn_gs.best_score['mse'],
        'cv_rmse': knn_gs.best_score['rmse'],
        'eval_mse': knn_eval_mse,
        'eval_rmse': knn_eval_rmse,
    },
    {
        'model': 'SVD (best)',
        'cv_mse': svd_gs.best_score['mse'],
        'cv_rmse': svd_gs.best_score['rmse'],
        'eval_mse': svd_eval_mse,
        'eval_rmse': svd_eval_rmse,
    }
])
display(comparison_metrics)

cols = ['user_id', 'rank', 'title', 'pred_rating']

compare_top10 = knn_top10[cols].merge(svd_top10[cols], on=['user_id','rank'], suffixes=('_knn','_svd')).sort_values(['user_id','rank'])


print("\n raccomandazioni e voti predetti a seconda del modello (KNN vs SVD)")
display(compare_top10.head(30))

,model,cv_mse,cv_rmse,eval_mse,eval_rmse
0,KNN (best),1.021999,1.010918,1.037704,1.018678
1,SVD (best),0.929667,0.964176,0.932949,0.965893



 raccomandazioni e voti predetti a seconda del modello (KNN vs SVD)


,user_id,rank,title_knn,pred_rating_knn,title_svd,pred_rating_svd
0,AE2324DZTYWS3LKNTEKJJZYHSQXQ,1,Heroes of Might and Magic III & IV Complete Co...,5.0,Devil May Cry,4.374909
1,AE2324DZTYWS3LKNTEKJJZYHSQXQ,2,Escape Rosecliff Island - PC,5.0,18 Wheels of Steel: Pedal to the Metal - PC,4.323215
2,AE2324DZTYWS3LKNTEKJJZYHSQXQ,3,Spartan: Total Warrior - PlayStation 2,5.0,Sega Genesis Core System 2 - Video Game Console,4.320335
3,AE2324DZTYWS3LKNTEKJJZYHSQXQ,4,NFL 2K3 - Xbox,5.0,Resident Evil 4 - Gamecube,4.318952
4,AE2324DZTYWS3LKNTEKJJZYHSQXQ,5,Tomee 256KB Memory Card for N64,5.0,Skylanders SWAP Force Sheep Wreck Island Adven...,4.272205
5,AE2324DZTYWS3LKNTEKJJZYHSQXQ,6,The Command and Conquer Collection - PC,5.0,Xbox Controller S- Green,4.234665
6,AE2324DZTYWS3LKNTEKJJZYHSQXQ,7,Whisper Fan 360 Green,5.0,Victor Vran: Overkill Edition - Nintendo Switch,4.216267
7,AE2324DZTYWS3LKNTEKJJZYHSQXQ,8,Commandos 2: Men of Courage - PC,5.0,Forza Motorsport,4.215761
8,AE2324DZTYWS3LKNTEKJJZYHSQXQ,9,Indiana Jones and the Emperor's Tomb - PC,5.0,Dragon Age: Origins Digital Deluxe Edition [Do...,4.210478
9,AE2324DZTYWS3LKNTEKJJZYHSQXQ,10,Namco Museum Vol. 1 (PlayStation),5.0,DualShock 4 Wireless Controller for PlayStatio...,4.201390
